In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
conn = sqlite3.connect("../db.sqlite3")
query = """
SELECT
    o.id AS order_id,
    o.datetime,
    p.name AS product,
    p.price,
    oi.quantity
FROM restaurant_order o
JOIN restaurant_orderitem oi ON o.id = oi.order_id
JOIN restaurant_product p ON oi.product_id = p.id
"""
df = pd.read_sql_query(query, conn)
df.to_csv("restaurant_orders.csv", index=False)
conn.close()

df = pd.read_csv("restaurant_orders.csv")
df.info()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
import matplotlib.pyplot as plt

top_products = df.groupby("product")["quantity"].sum().reset_index()
top_products = top_products.sort_values("quantity", ascending=False).head(10)

def format_label(pct, value):
    absolute = int(round(pct / 100. * sum(value)))
    return f"{pct:.1f}% ({absolute})"

plt.figure(figsize = (8,8))
plt.pie(
    top_products["quantity"],
    labels=top_products["product"],
    autopct=lambda pct: format_label(pct, top_products["quantity"]),)
plt.title("Top 10 positions in menu by quantity")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["item_price"] = df["quantity"] * df["price"]
top_price = df.groupby("product")["item_price"].sum().reset_index()
top_price = top_price.sort_values("item_price", ascending=False).head(10)

plt.figure(figsize = (8,8))
plt.pie(
    top_price["item_price"],
    labels=top_price["product"],
    autopct=lambda pct: format_label(pct, top_price["item_price"]),)
plt.title("Top 10 positions in menu by price")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["datetime"] = pd.to_datetime(df["datetime"])
df["order_hour"] = df["datetime"].dt.hour
order_hourly_income = df.groupby("order_hour")["item_price"].sum().reset_index()

plt.figure(figsize = (12,6))
plt.bar(
    order_hourly_income["order_hour"],
    order_hourly_income["item_price"], )
plt.title("Profit by order hours")
plt.xticks(range(24))
plt.xlabel("Order Hour")
plt.ylabel("Profit")
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["order_weekday"] = df["datetime"].dt.dayofweek
order_day_income = df.groupby("order_weekday")["item_price"].sum().reset_index()
order_day_income["order_weekday"] = order_day_income["order_weekday"].map(lambda x: weekdays[x])

plt.figure(figsize = (12,6))
plt.bar(
    order_day_income["order_weekday"],
    order_day_income["item_price"], )
plt.title("Profit by order weekdays")
plt.xlabel("Order Weekday")
plt.ylabel("Profit")
plt.show()